# Text State SQLite Monitor

Reads `state/text_downloader.sqlite` every 60 seconds and prints live status counts.
Stop the cell when done.

In [ ]:
import sqlite3
import time
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import clear_output

DB_PATH = Path('/content/video-virality-predictor/state/text_downloader.sqlite')
REFRESH_SECONDS = 60
RECENT_ROWS = 12

def read_snapshot(db_path: Path):
    uri = f'file:{db_path}?mode=ro'
    conn = sqlite3.connect(uri, uri=True, timeout=30)
    conn.execute('PRAGMA busy_timeout = 5000')

    total = conn.execute('SELECT COUNT(*) FROM processed_items').fetchone()[0]
    by_status = conn.execute('''
        SELECT status, COUNT(*) AS c
        FROM processed_items
        GROUP BY status
        ORDER BY c DESC, status ASC
    ''').fetchall()
    recent = conn.execute('''
        SELECT processed_at, video_id, status
        FROM processed_items
        ORDER BY processed_at DESC
        LIMIT ?
    ''', (RECENT_ROWS,)).fetchall()
    conn.close()
    return total, by_status, recent

while True:
    clear_output(wait=True)
    now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    print(f'[monitor] {now}')
    print(f'[monitor] db_path={DB_PATH}')

    if not DB_PATH.exists():
        print('[monitor] waiting for sqlite file...')
        time.sleep(REFRESH_SECONDS)
        continue

    try:
        total, by_status, recent = read_snapshot(DB_PATH)
        print(f'processed_total: {total}')
        print('\nstatus counts:')
        for status, count in by_status:
            pct = (count / total * 100.0) if total else 0.0
            print(f'  {status:35} {count:8d} ({pct:6.2f}%)')

        print('\nlatest rows:')
        for processed_at, video_id, status in recent:
            print(f'  {processed_at}  {video_id}  {status}')
    except Exception as exc:
        print(f'[monitor] read error: {exc}')

    time.sleep(REFRESH_SECONDS)
